# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name:\n  {metadata.name}\n\nDescription:\n  {metadata.description}")


## 2. Data Overview
Let's review available **record sets** and their fields. All references use `@id` for full reproducibility and correspondence to Croissant schema definitions.

In [ ]:
# List all record sets and their fields by @id
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print('Record sets in this dataset:')
for rs in metadata.to_json().get('recordSet', []):
    print(f"- Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '<none>')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print('  Fields:')
    for field in fields:
        fid = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
        print(f"    - {fid}")
    print()
if not record_set_ids:
    print("No record sets found, attempting to infer main record set...\n")
# If no record sets present, try to introspect distributions
distribution_ids = [d['@id'] for d in metadata.to_json().get('distribution', [])]
print('Distributions in this dataset:')
for d in metadata.to_json().get('distribution', []):
    print(f"- Distribution @id: {d['@id']}")

## 3. Data Extraction
Load data from a specific **record set** into a DataFrame for analysis. As this dataset does not explicitly define Croissant record sets, we'll try to load records directly from the primary distribution. All entities are referenced by `@id`.

In [ ]:
# Use the distribution @id to load records (as there's no explicit recordSet)
main_distribution_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
# Try to load records using mlcroissant (most Croissant datasets support this)

try:
    records = list(dataset.records())  # falls back to implicit main table if record_set is not provided
except Exception as e:
    print("Failed to load records using default mlcroissant.records():", e)
    records = []
df = pd.DataFrame(records) if records else pd.DataFrame()

print("Extracted fields:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by threshold, normalizing a numeric field, and grouping by an attribute. All field references use their Croissant `@id`.

Below we attempt to use common protein mass spectrometry fields such as `cr:MW` (Molecular Weight), `cr:abundance`, or similar. You should adapt the field `@id` to match the columns present above. If unsure, inspect the extracted field list in the previous cell.

In [ ]:
# Pick a numeric field for demonstration. Replace 'cr:MW' with a field present in your df.columns, e.g. 'MW', 'MolecularWeight', etc.
# Example field IDs (to be adjusted to your actual column names):
numeric_field_id = None
for col in df.columns:
    if ('weight' in col.lower()) or (col.lower() in ['mw', 'molecular_weight', 'cr:MW']):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # As fallback, pick the first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if numeric_field_id:
    print(f"Using numeric field `@id`: {numeric_field_id}")
    # Remove non-numeric or missing values
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > 10000]
    print(f"Filtered records with {numeric_field_id} > 10000 (arbitrary molecular weight threshold):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Attempt grouping by a common protein annotation field, e.g., 'cr:accession', 'accession', etc.
    group_field = None
    for candidate in ['cr:accession', 'accession', 'protein_id']:
        if candidate in filtered_df.columns:
            group_field = candidate
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print('No numeric (e.g., molecular weight) field found in columns.')

## 5. Visualization
Visualize distributions or relationships between fields using matplotlib and seaborn (if present). Field names correspond to the `@id` as in previous steps.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
else:
    print('No numeric field or data available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a mass spectrometry dataset on extracellular vesicles using the `mlcroissant` library. All field and record set references use their Croissant `@id`s. For further analysis, refer to the [Croissant specification](https://mlcommons.github.io/croissant/).